# Auto Marker Machine

In [ ]:
import pandas as pd

# initializing marker
file = r"C:\Users\GG\Desktop\marker blso baru.xlsx"

marker = pd.read_excel(file, 
                       sheet_name = "Marker")
marker = pd.pivot_table(marker, 
                        index = "Well", 
                        columns = "Surface", 
                        values = "MD")
marker = marker.reset_index().rename_axis(None, axis = 1)

# completion event
perfo = pd.read_excel(file, 
                      sheet_name = "Komplesi")

# list of sands
sand = pd.read_excel(file,
                       sheet_name = "Sand")['Marker'].tolist()

# merging the marker and perfo data
perfo_wells = perfo.merge(marker,
                          how = 'left', 
                          left_on = 'WELL', 
                          right_on = 'Well')

perfo_wells = perfo_wells[['WELL','DATE','Perf Status','Perf Top (ftMD)','Perf Base (ftMD)']+sand]

In [ ]:
# Markering Telisa and Sihapas
import warnings
import numpy as np
warnings.filterwarnings("ignore")

# blank dataframe to contain marker from top completion and bottom completion
df = pd.DataFrame()

# start markering!
for x,y in list(zip(['Perf Top (ftMD)', 'Perf Base (ftMD)'], ['TOP_WAY', 'BOTTOM_WAY'])):

    # initialising iterator
    i = 0

    # looping through all wells
    for well in list(perfo_wells['WELL'].unique()):
        
        # initialising temporary dataframe
        temp = perfo_wells[perfo_wells['WELL'] == well]
        temp = temp.dropna(axis = 1)
        sand_list = list(temp.columns[5:])
        
        j = 0
        while j < len(temp):

            for k in range(0, len(sand_list)):
                
                # adressing one marker only
                if len(sand_list) == 1:
                    if temp[x][i+j] >= temp[sand_list[k]][i+j]:
                        df.loc[i+j,y] = str(sand[sand.index(sand_list[k])])
                        break
                    else:
                        df.loc[i+j,y] = str(sand[sand.index(sand_list[k]) - 1])
                        break
                else:
                    # addressing first sand
                    if sand_list[k] == sand_list[0]:
                        if temp[x][i+j] < temp[sand_list[k]][i+j]:
                            df.loc[i+j,y] = str(sand[sand.index(sand_list[k]) - 1])
                            break
                
                    # addressing last sand
                    elif sand_list[k] == sand_list[-1]:

                        if temp[x][i+j] >= temp[sand_list[k]][i+j]:
                            df.loc[i+j,y] = str(sand_list[sand_list.index(sand_list[k])])
                            break
                        else:
                            df.loc[i+j,y] = str(sand_list[sand_list.index(sand_list[k]) - 1])
                            break
                    
                    # addressing all sand except first and last sand
                    else:
                        if temp[x][i+j] < temp[sand_list[k]][i+j]:
                            df.loc[i+j,y] = str(sand_list[sand_list.index(sand_list[k]) - 1])
                            break
            
            j += 1 
            
        i += j

# dropping marker columns
perfo_wells['TOP_WAY'] = df['TOP_WAY']
perfo_wells['BOTTOM_WAY'] = df['BOTTOM_WAY']
perfo_wells['MARKER'] = np.nan # containing final marker

i = 0

# markering final marker
for well in list(perfo_wells['WELL'].unique()):
    
    # initialising temporary dataframe
    temp = perfo_wells[perfo_wells['WELL'] == well]
    temp = temp.dropna(axis = 1)
    if 'TOP_WAY' not in temp.columns or 'BOTTOM_WAY' not in temp.columns:
        i += len(temp)
        continue
    temp.drop(['TOP_WAY', 'BOTTOM_WAY'], axis=1, inplace=True)
    sand_list = list(temp.columns[5:])
    
    j = 0
    while j < len(temp):
        
        if (perfo_wells['TOP_WAY'][i+j] == 'T_PM2300' and perfo_wells['BOTTOM_WAY'][i+j] == 'T_PM2300'):
            perfo_wells['MARKER'][i+j] = ['T_TE600']
        elif (perfo_wells['TOP_WAY'][i+j] == 'T_PM2300'):
            perfo_wells['MARKER'][i+j] = sand_list[0:sand_list.index(perfo_wells['BOTTOM_WAY'][i+j])+1]
        elif (
            (perfo_wells['TOP_WAY'][i+j] not in sand_list and perfo_wells['BOTTOM_WAY'][i+j] not in sand_list) or 
            (perfo_wells['TOP_WAY'][i+j] not in sand_list or perfo_wells['BOTTOM_WAY'][i+j] not in sand_list)
            ):
            perfo_wells['MARKER'][i+j] = sand[sand.index(perfo_wells['TOP_WAY'][i+j]):sand.index(perfo_wells['BOTTOM_WAY'][i+j])+1]
        else:
            perfo_wells['MARKER'][i+j] = sand_list[sand_list.index(perfo_wells['TOP_WAY'][i+j]):sand_list.index(perfo_wells['BOTTOM_WAY'][i+j])+1]

        j += 1
    
    i += j

In [ ]:
# function to process the perforation and squeeze events
def squeeze_machine(tp,bp,pf,pd,ts,bs,sf,sd,marker_temp):
    
    a = 0
    # looping through every squeezed event   
    while a < len(sd):

        #checking each squeeze event to each perforation event
        b = 0
        while b < len(pd):
            
            # handling partial squeeze
            if (sd[a] >= pd[b]):
                
                # changing top perforation and deleting squeezed sand
                if ((ts[a] == tp[b]) and  bs[a] < bp[b]):
                    tp[b] = bs[a]
                    if (len(pf[b])==len(sf[a])):
                        pf[b] = [pf[b][-1]]
                    elif (len(pf[b]) > len(sf[a])):
                        if tp[b] < marker_temp.loc[0, pf[b][len(sf[a])]]:
                            pf[b] = [pf[b][len(sf[a])-1]] + [x for x in pf[b] if x not in sf[a]]
                        else:
                            pf[b] = [x for x in pf[b] if x not in sf[a]]

                # changing bottom perforation and deleting squeezed sand
                elif ((ts[a] > tp[b]) and  bs[a] == bp[b]):
                    bp[b] = ts[a]
                    if (len(pf[b])==len(sf[a])):
                        pf[b] = [pf[b][0]] 
                    elif (len(pf[b]) > len(sf[a])):
                        if bp[b] > marker_temp.loc[0, pf[b][len(sf[a])]]:
                            pf[b] = [pf[b][len(sf[a])]] + [x for x in pf[b] if x not in sf[a]]
                        else:
                            pf[b] = [x for x in pf[b] if x not in sf[a]]
       
            # deleting perforation event (date, top perforation, bottom perforation and sand) that squeezed
            if ((ts[a] <= tp[b] and bs[a] >= bp[b]) 
                and sd[a] >= pd[b]):
                del pf[b]
                del tp[b]
                del bp[b]
                del pd[b] 
            else:
                b += 1

        a += 1
    
    return tp,bp,pf

# function to count perforated sand at certain event of production
def count_sand(res, pf):

    if len(pf) > 0:
        
        x = 0
        for i in range(0, len(pf)):
            if res in pf[i]:
                x += 1        
        return x

    else:
        return 0
    
def writing_perforation (production, sands, i):

    x = 0
    for key, value in sands.items():
        if value > 0:
            production.loc[i, key]= "p"
        x += 1

In [ ]:
# load data
marker = perfo_wells
prod = pd.read_excel(file, sheet_name = "Produksi")

# create blank dataframe to store the result
final = pd.DataFrame()

# starting by looping through all wells from production data
for uwi in list(prod['WELL'].unique()):

    # checking wether the selected well is in the marker data or not
    if uwi in marker['WELL'].unique() and marker[marker['WELL'] == uwi]['MARKER'].notna().any():
        
        # partiliazing marker data 
        tempmarker = marker[marker['WELL']==uwi]
        tempmarker = tempmarker.dropna(axis = 1)
        tempmarker = tempmarker.reset_index(drop=True)

        # partializing production data
        temp_prod = prod[prod['WELL']==uwi]
        temp_prod = temp_prod.reset_index(drop=True)

        # looping through selected well from production data
        index = 0
        
        while index < len(temp_prod):

            # initialize starting variable
            x = 0

            # initialize empty array (perforation sand, squeeze sand, top perforation, bottom perforation, _
            # _ top squeeze, bottom squeeze, perforation date, squeeze date)
            PF,SF,TP,BP,TS,BS,PD,SD = [],[],[],[],[],[],[],[] 
            
            # checking the production data through marker data
            while x < len(tempmarker):

                # dates in every single timestep in production data is compared to all of the dates data that meet the condition
                if (
                    (temp_prod['DATE'][index].year > tempmarker['DATE'][x].year) or 
                    ((temp_prod['DATE'][index].year == tempmarker['DATE'][x].year) and 
                    (temp_prod['DATE'][index].month >= tempmarker['DATE'][x].month))
                    ):

                    # perforation event 
                    if tempmarker['Perf Status'][x] == "perforation":
                        
                        # appending perforation event data
                        PF.append(tempmarker['MARKER'][x])
                        TP.append(tempmarker['Perf Top (ftMD)'][x])
                        BP.append(tempmarker['Perf Base (ftMD)'][x])
                        PD.append(tempmarker['DATE'][x])

                    # squeeze event
                    elif tempmarker['Perf Status'][x] == "squeeze":

                        # appending squeeze event data
                        SF.append(tempmarker['MARKER'][x])
                        TS.append(tempmarker['Perf Top (ftMD)'][x])
                        BS.append(tempmarker['Perf Base (ftMD)'][x])
                        SD.append(tempmarker['DATE'][x])
                    
                    else:
                        pass
        
                    x += 1

                else:
                    break

            # squeeze handling 
            squeeze_machine(TP,BP,PF,PD,TS,BS,SF,SD,tempmarker)

            # print the result in the terminal
            print(f'Well : {uwi} | index : {index} | {x} |{PF} | {list(zip(TP,BP))}')

            # creating dictionary that hold number of perforated sands
            sands = {}
            for baso in sand:
                sands[baso] = count_sand(baso, PF)

            # write "p" on the dataframe to mark perforated sand
            writing_perforation(temp_prod, sands, index)
            
            index += 1

    else:
        continue
    
    # concating the result of each well after the "p" writing process
    final = pd.concat([final,temp_prod])

# Splitting Machine

In [ ]:
# newest version
# splitting machine 
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# enter the path/directory of the format data splitting (depends on where you put the file)
file_path = r"/home/grhaputra/marker blso baru.xlsx"

# sand data
sands = pd.read_excel(file_path, sheet_name = "Sand")['Marker'].tolist()

# lumping data
lumping = pd.read_excel(file_path, sheet_name = "Lumping").set_index("Zone log")

# perforation data
perfo = pd.read_excel(file_path, sheet_name = "P (monthly)")

# well data
well = pd.read_excel(file_path, sheet_name = "Well")

# copy the existing dataframe for split
perfo_column = 8 # number of column before the sand list in markered production sheet

 # create new dataframe for splitting and copy the original columns
df_split = perfo[perfo.columns[:perfo_column]]
df = df_split.copy()

# create new columns for splitting
for s in sands:
    df["OIL_" + s] = np.nan
    df["WATER_" + s] = np.nan
    df["WINJ_" + s] = np.nan

row_count = 0
i = 0

for index in well.index:
    
    while i < len(perfo):
        
        if str(perfo['WELL'][i]) == str(well['Well'][index]):
            
            # define total_kh initial
            total_kh = 0

            # calculate the total kh for each time step
            for sand in perfo.columns[perfo_column:]:
                if perfo.loc[row_count, sand] == 'p':
                    total_kh += lumping[str(well['Well'][index])][sand]

            # calculate the volume fraction of each sand for each time step
            for sand in perfo.columns[perfo_column:]:
                if total_kh > 0:
                    if perfo.loc[row_count, sand] == 'p':
                        df.loc[row_count, 'OIL_'+sand] = float(perfo['OIL'][i])*float(lumping[str(well['Well'][index])][sand])/total_kh
                        df.loc[row_count, 'WATER_'+sand] = float(perfo['WATER'][i])*float(lumping[str(well['Well'][index])][sand])/total_kh
                        df.loc[row_count, 'WINJ_'+sand] = float(perfo['WINJ'][i])*float(lumping[str(well['Well'][index])][sand])/total_kh
                    else:
                        df.loc[row_count, 'OIL_'+sand] = 0
                        df.loc[row_count, 'WATER_'+sand] = 0
                        df.loc[row_count, 'WINJ_'+sand] = 0
            
            row_count += 1
            i += 1

            print(str(well['Well'][index]) + "|" + str(i))

        else:

            break

df.to_excel(r"/home/grhaputra/New BLSO Splitting Bulk with New Wells Updated 2.xlsx", index=False)

BLSO00001|1
BLSO00001|2
BLSO00001|3
BLSO00001|4
BLSO00001|5
BLSO00001|6
BLSO00001|7
BLSO00001|8
BLSO00001|9
BLSO00001|10
BLSO00001|11
BLSO00001|12
BLSO00001|13
BLSO00001|14
BLSO00001|15
BLSO00001|16
BLSO00001|17
BLSO00001|18
BLSO00001|19
BLSO00001|20
BLSO00001|21
BLSO00001|22
BLSO00001|23
BLSO00001|24
BLSO00001|25
BLSO00001|26
BLSO00001|27
BLSO00001|28
BLSO00001|29
BLSO00001|30
BLSO00001|31
BLSO00001|32
BLSO00001|33
BLSO00001|34
BLSO00001|35
BLSO00001|36
BLSO00001|37
BLSO00001|38
BLSO00001|39
BLSO00001|40
BLSO00001|41
BLSO00001|42
BLSO00001|43
BLSO00001|44
BLSO00001|45
BLSO00001|46
BLSO00001|47
BLSO00001|48
BLSO00001|49
BLSO00001|50
BLSO00001|51
BLSO00001|52
BLSO00001|53
BLSO00001|54
BLSO00001|55
BLSO00001|56
BLSO00001|57
BLSO00001|58
BLSO00001|59
BLSO00001|60
BLSO00001|61
BLSO00001|62
BLSO00001|63
BLSO00001|64
BLSO00001|65
BLSO00001|66
BLSO00001|67
BLSO00001|68
BLSO00001|69
BLSO00001|70
BLSO00001|71
BLSO00001|72
BLSO00001|73
BLSO00001|74
BLSO00001|75
BLSO00001|76
BLSO00001|77
BLSO0000

In [4]:
# newest version
# splitting machine 
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# enter the path/directory of the format data splitting (depends on where you put the file)
file_path = r"/home/grhaputra/Downloads/TEMPLATE.xlsx"

# sand data
sands = pd.read_excel(file_path, sheet_name = "Sand")['Marker'].tolist()

# lumping data
lumping = pd.read_excel(file_path, sheet_name = "Lumping GG").set_index("Zone log")

# perforation data
perfo = pd.read_excel(file_path, sheet_name = "P (monthly)")

# well data
well = pd.read_excel(file_path, sheet_name = "Well")

# copy the existing dataframe for split
perfo_column = 7 # number of column before the sand list in markered production sheet

 # create new dataframe for splitting and copy the original columns
df_split = perfo[perfo.columns[:perfo_column]]
df = df_split.copy()

# create new columns for splitting
for s in sands:
    df["OIL_" + s] = np.nan
    df["WATER_" + s] = np.nan
    df["GAS_" + s] = np.nan
    df["WINJ_" + s] = np.nan

row_count = 0
i = 0

for index in well.index:
    
    while i < len(perfo):
        
        if str(perfo['Well'][i]) == str(well['Well'][index]):
            
            # define total_kh initial
            total_kh = 0

            # calculate the total kh for each time step
            for sand in perfo.columns[perfo_column:]:
                if perfo.loc[row_count, sand] == 'P':
                    total_kh += lumping[str(well['Well'][index])][sand]

            # calculate the volume fraction of each sand for each time step
            for sand in perfo.columns[perfo_column:]:
                if total_kh > 0:
                    if perfo.loc[row_count, sand] == 'P':
                        df.loc[row_count, 'OIL_'+sand] = float(perfo['OIL'][i])*float(lumping[str(well['Well'][index])][sand])/total_kh
                        df.loc[row_count, 'WATER_'+sand] = float(perfo['WATER'][i])*float(lumping[str(well['Well'][index])][sand])/total_kh
                        df.loc[row_count, 'GAS_'+sand] = float(perfo['GAS'][i])*float(lumping[str(well['Well'][index])][sand])/total_kh
                        df.loc[row_count, 'WINJ_'+sand] = float(perfo['WINJ'][i])*float(lumping[str(well['Well'][index])][sand])/total_kh
                    else:
                        df.loc[row_count, 'OIL_'+sand] = 0
                        df.loc[row_count, 'WATER_'+sand] = 0
                        df.loc[row_count, 'GAS_'+sand] = 0
                        df.loc[row_count, 'WINJ_'+sand] = 0
            
            row_count += 1
            i += 1

            print(str(well['Well'][index]) + "|" + str(i))

        else:

            break

df.to_excel(r"/home/grhaputra/Splitting Buat Dito.xlsx", index=False)

L5A-51|1
L5A-51|2
L5A-51|3
L5A-51|4
L5A-51|5
L5A-51|6
L5A-51|7
L5A-51|8
L5A-51|9
L5A-51|10
L5A-51|11
L5A-51|12
L5A-51|13
L5A-51|14
L5A-51|15
L5A-51|16
L5A-51|17
L5A-51|18
L5A-51|19
L5A-51|20
L5A-51|21
L5A-51|22
L5A-51|23
L5A-51|24
L5A-51|25
L5A-51|26
L5A-51|27
L5A-51|28
L5A-51|29
L5A-51|30
L5A-51|31
L5A-51|32
L5A-51|33
L5A-51|34
L5A-51|35
L5A-51|36
L5A-51|37
L5A-51|38
L5A-51|39
L5A-51|40
L5A-51|41
L5A-51|42
L5A-51|43
L5A-51|44
L5A-51|45
L5A-51|46
L5A-51|47
L5A-51|48
L5A-51|49
L5A-51|50
L5A-51|51
L5A-51|52
L5A-51|53
L5A-51|54
L5A-51|55
L5A-51|56
L5A-51|57
L5A-51|58
L5A-51|59
L5A-51|60
L5A-51|61
L5A-51|62
L5A-51|63
L5A-51|64
L5A-51|65
L5A-51|66
L5A-51|67
L5A-51|68
L5A-51|69
L5A-51|70
L5A-51|71
L5A-51|72
L5A-51|73
L5A-51|74
L5A-51|75
L5A-51|76
L5A-51|77
L5A-51|78
L5A-51|79
L5A-51|80
L5A-51|81
L5A-51|82
L5A-51|83
L5A-51|84
L5A-51|85
L5A-51|86
L5A-51|87
L5A-51|88
L5A-51|89
L5A-51|90
L5A-51|91
L5A-51|92
L5A-51|93
L5A-51|94
L5A-51|95
L5A-51|96
L5A-51|97
L5A-51|98
L5A-51|99
L5A-51|100
L5A-51|1

# Summary Splitting Machine

In [ ]:
# summary
import pandas as pd

# enter the path/directory of the format data splitting (depends on where you put the file)
file_path = r"/home/grhaputra/marker blso baru.xlsx"

# sand data
sands = pd.read_excel(file_path, sheet_name = "Sand")['Marker'].tolist()

summary = pd.DataFrame(df)

# Dynamically extract unique suffixes from the column names
suffixes = sands

# Initialize summary dictionary
# summary_dict = {'Sand': [], 'Oil': [], 'Water': [], 'Water Inj': []}
summary_dict = {'Sand': [], 'Oil': []}

# Iterate over each suffix to calculate the sum for oil and water
for suffix in suffixes:
    oil_columns = [col for col in summary.columns if col.startswith('OIL_') and col.endswith(suffix)]
    # water_columns = [col for col in summary.columns if col.startswith('WATER_') and col.endswith(suffix)]
    # water_inj_columns = [col for col in summary.columns if col.startswith('WINJ_') and col.endswith(suffix)]
    
    # Add the results to the summary dictionary
    summary_dict['Sand'].append(suffix)
    summary_dict['Oil'].append(summary[oil_columns].sum().sum())   # Sum across all oil columns for the suffix
    # summary_dict['Water'].append(summary[water_columns].sum().sum()) # Sum across all water columns for the suffix
    # summary_dict['Water Inj'].append(summary[water_inj_columns].sum().sum()) # Sum across all water inj columns for the suffix

# Create the summary DataFrame
summary_final = pd.DataFrame(summary_dict)

# Display the summary
display(summary_final)

,Sand,Oil
0,T_TE600,5.704360e+06
1,T60,1.747271e+07
2,T80,4.786119e+06
3,T90,2.344487e+06
4,T100,1.249926e+07
5,T_TE760,1.332314e+05
6,T_TE810,3.674660e+04
7,T_TE830,4.168226e+04
8,T_D900U,4.783440e+07
9,T_D900L,2.343778e+07


In [10]:
import pandas as pd

df = pd.read_excel(r"/home/grhaputra/[ Project Perminyakan ]/Lapangan Petapahan/Splitting Demo FIle.xlsx", sheet_name = "Sheet3")
df_melt = df.melt(id_vars = ["Zone"], var_name = "Well", value_name = "KH")
df_melt.to_excel(r"/home/grhaputra/[ Project Perminyakan ]/Lapangan Petapahan/Melting.xlsx", index = False)